# 4 · Control Flow and Loops

*Language basics* · 9 exercises

Loops, ranges and branching, including for-else and match.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-4) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-4)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 4.1 · FizzBuzz  `Core`

**Task.** Write `fizzbuzz(n)` returning a list for 1..n: `"Fizz"` for multiples of 3, `"Buzz"` for 5, `"FizzBuzz"` for both, otherwise the number as a string.

**Example.** `fizzbuzz(5)` -> `['1', '2', 'Fizz', '4', 'Buzz']`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-1)

In [ ]:
# 4.1 FizzBuzz
def fizzbuzz(n):
    raise NotImplementedError

In [ ]:
def test_ex_4_1():
    assert fizzbuzz(5) == ['1', '2', 'Fizz', '4', 'Buzz']
    assert fizzbuzz(15)[-1] == 'FizzBuzz'


test_ex_4_1()
print('4.1 passed')

### 4.2 · for ... else  `Core`

**Task.** Write `first_multiple(nums, k)` returning the index of the first multiple of k, or -1, using a `for` loop with an `else` clause instead of a flag variable.

**Example.** `first_multiple([5, 7, 12, 9], 3)` -> `2`; `first_multiple([5, 7], 3)` -> `-1`

**Big O.** O(N).

**Note.** The `else` of a loop runs only when the loop finished without `break`.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-2)

In [ ]:
# 4.2 for ... else
def first_multiple(nums, k):
    raise NotImplementedError

In [ ]:
def test_ex_4_2():
    assert first_multiple([5, 7, 12, 9], 3) == 2
    assert first_multiple([5, 7], 3) == -1
    assert first_multiple([], 3) == -1


test_ex_4_2()
print('4.2 passed')

### 4.3 · while loop: Collatz steps  `Core`

**Task.** Write `collatz_steps(n)`: halve even numbers, map odd n to 3n + 1, and count steps until n is 1.

**Example.** `collatz_steps(6)` -> `8`; `collatz_steps(1)` -> `0`

**Big O.** No known bound; that is why it needs `while`, not `for`.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-3)

In [ ]:
# 4.3 while loop: Collatz steps
def collatz_steps(n):
    raise NotImplementedError

In [ ]:
def test_ex_4_3():
    assert collatz_steps(6) == 8
    assert collatz_steps(1) == 0


test_ex_4_3()
print('4.3 passed')

### 4.4 · Nested loops: multiplication table  `Core`

**Task.** Write `times_table(n)` returning an n by n list of lists where row i, column j holds (i + 1) * (j + 1).

**Example.** `times_table(3)` -> `[[1, 2, 3], [2, 4, 6], [3, 6, 9]]`

**Big O.** O(N^2).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-4)

In [ ]:
# 4.4 Nested loops: multiplication table
def times_table(n):
    raise NotImplementedError

In [ ]:
def test_ex_4_4():
    assert times_table(3) == [[1, 2, 3], [2, 4, 6], [3, 6, 9]]


test_ex_4_4()
print('4.4 passed')

### 4.5 · range with a step  `Core`

**Task.** Write `evens_down(n)` returning the even numbers from n (or n - 1 if n is odd) down to 0, using a single `range` call.

**Example.** `evens_down(7)` -> `[6, 4, 2, 0]`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-5)

In [ ]:
# 4.5 range with a step
def evens_down(n):
    raise NotImplementedError

In [ ]:
def test_ex_4_5():
    assert evens_down(7) == [6, 4, 2, 0]
    assert evens_down(6) == [6, 4, 2, 0]


test_ex_4_5()
print('4.5 passed')

### 4.6 · enumerate and zip  `Core`

**Task.** Write `rank_lines(names, scores)` returning lines like `"1. ann: 90"` using `zip` to pair the lists and `enumerate(..., start=1)` for the rank.

**Example.** `rank_lines(["ann", "bo"], [90, 75])` -> `['1. ann: 90', '2. bo: 75']`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-6)

In [ ]:
# 4.6 enumerate and zip
def rank_lines(names, scores):
    raise NotImplementedError

In [ ]:
def test_ex_4_6():
    assert rank_lines(['ann', 'bo'], [90, 75]) == ['1. ann: 90', '2. bo: 75']


test_ex_4_6()
print('4.6 passed')

### 4.7 · Pattern printing  `Core`

**Task.** Write `pyramid(n)` returning n lines of a centred star pyramid, the classic warm-up.

**Example.** `pyramid(3)` -> `['  *', ' ***', '*****']`

**Big O.** O(N^2) characters.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-7)

In [ ]:
# 4.7 Pattern printing
def pyramid(n):
    raise NotImplementedError

In [ ]:
def test_ex_4_7():
    assert pyramid(3) == ['  *', ' ***', '*****']


test_ex_4_7()
print('4.7 passed')

### 4.8 · break and continue  `Core`

**Task.** Write `sum_until_negative(nums)`: add numbers in order, skip zeros with `continue`, and stop at the first negative with `break`.

**Example.** `sum_until_negative([4, 0, 3, -1, 10])` -> `7`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-8)

In [ ]:
# 4.8 break and continue
def sum_until_negative(nums):
    raise NotImplementedError

In [ ]:
def test_ex_4_8():
    assert sum_until_negative([4, 0, 3, -1, 10]) == 7
    assert sum_until_negative([]) == 0


test_ex_4_8()
print('4.8 passed')

### 4.9 · match statement  `Practice`

**Task.** Write `calc(op, a, b)` with a `match` statement for `"+"`, `"-"`, `"*"`, `"/"` (integer floor division) and a default case that raises `ValueError`.

**Example.** `calc("*", 6, 7)` -> `42`; `calc("/", 7, 2)` -> `3`

**Big O.** O(1). Needs Python 3.10 or newer; check your judge's version.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-4-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-4-9)

In [ ]:
# 4.9 match statement
def calc(op, a, b):
    raise NotImplementedError

In [ ]:
def test_ex_4_9():
    assert calc('*', 6, 7) == 42
    assert calc('/', 7, 2) == 3
    assert calc('-', 1, 4) == -3
    with pytest.raises(ValueError):
        calc('^', 1, 2)


test_ex_4_9()
print('4.9 passed')